In [1]:
!pip install -q \
langchain \
langchain-community \
langchain-huggingface \
transformers \
accelerate \
sentence-transformers \
faiss-cpu \
chromadb


In [2]:
from google.colab import drive
import os
drive.mount('/content/drive')

os.chdir('/content/drive/MyDrive/ufc-rag-proj')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import os
from pathlib import Path
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
import torch

PROJECT_ROOT = Path('/content/drive/MyDrive/ufc-rag-proj')
DOCS_DIR = PROJECT_ROOT / "documents"
CHROMA_PATH = PROJECT_ROOT / 'vectorstore' / 'chroma_db'

print(f'Fight docs {len(list((DOCS_DIR/'fights').glob('*.txt')))}')
print(f'Event docs {len(list((DOCS_DIR/'events').glob('*.txt')))}')
print(f'Fighter docs {len(list((DOCS_DIR/'fighters').glob('*.txt')))}')


/tmp/ipykernel_14339/3903011213.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader


Fight docs 8436
Event docs 768
Fighter docs 2685


In [4]:
loader = DirectoryLoader(
    str(DOCS_DIR),
    glob="**/*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": 'utf-8'},
    show_progress=True
)

documents = loader.load()
print(f"Total documents loaded {len(documents)}")

100%|██████████| 11889/11889 [00:34<00:00, 339.69it/s]

Total documents loaded 11889


In [5]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size= 800,
    chunk_overlap=100,
    length_function= len
)

chunks = text_splitter.split_documents(documents)
print(len(chunks))

12553


In [6]:
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={"device":"cuda"},
    encode_kwargs={"normalize_embeddings":True}
)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [7]:
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory=str(CHROMA_PATH)
)
print("Vector store created succesfully")
print(f"Saved at {CHROMA_PATH}")

Vector store created succesfully
Saved at /content/drive/MyDrive/ufc-rag-proj/vectorstore/chroma_db
